# Convertible note offering — bull put spread

Thesis: convertible note offerings generate mechanical stock pressure from arb hedge flows. 
After the hedge-driven drop, implied volatility is elevated. Enter a bull put spread on day 1 or 2 close, 
2–4 weeks to expiry, short strike near spot, long strike ~1 ATR below. Exit at half max profit or day 10.


## 1 · Imports + API client (cached, rate-limit-aware)


In [186]:
import os, json, time, hashlib, re
from dataclasses import dataclass, field
from pathlib import Path
from getpass import getpass

import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday, USMartinLutherKingJr,
                                    USPresidentsDay, GoodFriday, USMemorialDay, USLaborDay, USThanksgivingDay)
from pandas.tseries.offsets import CustomBusinessDay

BASE_URL = "https://api.massive.com"


def load_api_key(name: str = "MASSIVE_API_KEY") -> str:
    """Environment first, then a `.env` file beside the notebook, then a prompt. Never paste a key into a cell."""
    key = (os.environ.get(name) or "").strip()
    env_file = Path(".env")
    if not key and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith(f"{name}="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not key:
        key = getpass(f"{name} (https://massive.com/dashboard/keys): ").strip()
    return key


API_KEY = load_api_key()
SESSION = requests.Session()
SESSION.headers["Authorization"] = f"Bearer {API_KEY}"
# Pricing runs PRICE_WORKERS events at once, so the connection pool must be bigger than the
# default 10 or every worker blocks on a fresh TCP+TLS handshake.
SESSION.mount("https://", requests.adapters.HTTPAdapter(pool_maxsize=32, pool_connections=32))

CACHE_DIR = Path(".massive_cache")
CACHE_DIR.mkdir(exist_ok=True)


def api_get(path_or_url: str, params: dict | None = None) -> dict:
    """GET one page from the Massive REST API. Cached on disk by the full URL, so reruns are free."""
    url = path_or_url if path_or_url.startswith("http") else BASE_URL + path_or_url
    full_url = requests.Request("GET", url, params=params).prepare().url
    cache_file = CACHE_DIR / (hashlib.sha1(full_url.encode()).hexdigest() + ".json")
    if cache_file.exists():
        try:
            txt = cache_file.read_text()
            if txt.strip():
                return json.loads(txt)
        except (json.JSONDecodeError, ValueError):
            cache_file.unlink(missing_ok=True)
    for attempt in range(10):
        resp = SESSION.get(full_url, timeout=60)
        if resp.status_code in (429, 500, 502, 503, 504):
            retry_after = resp.headers.get("Retry-After", "")
            time.sleep(float(retry_after) if retry_after.isdigit() else min(2 ** attempt, 20))
            continue
        break
    resp.raise_for_status()
    try:
        payload = resp.json()
    except ValueError:
        print(f"WARNING: non-JSON response from {url} (status {resp.status_code})")
        payload = {}
    tmp = cache_file.with_suffix(".tmp")          # atomic: an interrupted or parallel run
    tmp.write_text(json.dumps(payload))           # never leaves a half-written page behind
    tmp.replace(cache_file)
    return payload


def api_get_all(path: str, params: dict | None = None, max_pages: int = 500) -> list[dict]:
    """Follow `next_url` pagination and return every result row."""
    payload = api_get(path, params)
    rows = list(payload.get("results") or [])
    pages = 1
    while payload.get("next_url") and pages < max_pages:
        payload = api_get(payload["next_url"])
        rows.extend(payload.get("results") or [])
        pages += 1
    return rows


print(f"API key loaded (ends {API_KEY[-4:]}); cache at {CACHE_DIR.resolve()}")

API key loaded (ends bDot); cache at C:\Users\yash2\Desktop\OpenSource\massiveandslightlypassive\.massive_cache


## 2 · Trading calendar (no stock feed in this challenge)


In [187]:
class NYSEHolidays(AbstractHolidayCalendar):
    """NYSE full-day closures. Differs from the federal calendar: Good Friday closed, Columbus and
    Veterans Day open, no observance when New Year's Day falls on a Saturday."""
    rules = [
        Holiday("New Year's Day", month=1, day=1,
                observance=lambda d: d + pd.Timedelta(days=1) if d.weekday() == 6 else d),
        USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay,
        Holiday("Juneteenth", month=6, day=19, start_date="2022-01-01", observance=nearest_workday),
        Holiday("Independence Day", month=7, day=4, observance=nearest_workday),
        USLaborDay, USThanksgivingDay,
        Holiday("Christmas Day", month=12, day=25, observance=nearest_workday),
        Holiday("National day of mourning, President Carter", year=2025, month=1, day=9),
    ]


def trading_sessions(start, end) -> pd.DatetimeIndex:
    holidays = NYSEHolidays().holidays(pd.Timestamp(start) - pd.Timedelta(days=7), pd.Timestamp(end) + pd.Timedelta(days=7))
    return pd.bdate_range(start, end, freq=CustomBusinessDay(holidays=holidays))


CAL = trading_sessions("2021-06-01", "2027-12-31")
TODAY = pd.Timestamp.today().normalize()
LAST_SESSION = CAL[CAL.searchsorted(TODAY, side="right") - 1]


def session_on_or_after(day) -> pd.Timestamp:
    return CAL[CAL.searchsorted(pd.Timestamp(day), side="left")]


def session_before(day) -> pd.Timestamp:
    return CAL[CAL.searchsorted(pd.Timestamp(day), side="left") - 1]


def sessions_between(a, b) -> int:
    """Number of sessions strictly after `a` up to and including `b`."""
    return int(CAL.searchsorted(pd.Timestamp(b), side="right") - CAL.searchsorted(pd.Timestamp(a), side="right"))


print(f"{len(CAL)} sessions in the calendar; last completed session {LAST_SESSION.date()}. "
      f"Sessions per year: " + ", ".join(f"{y}: {((CAL.year == y)).sum()}" for y in range(2022, 2027)))

1655 sessions in the calendar; last completed session 2026-10-02. Sessions per year: 2022: 251, 2023: 250, 2024: 252, 2025: 250, 2026: 251


## 3 · Configuration — convertible note offering, bull put spread


In [188]:
# ---- Convertible note offering strategy ----------------------------------------------------
# Item codes for convertible note offerings in 8-Ks
CONVERTIBLE_ITEM_CODES = ["1.01", "2.03"]

# Keywords to identify convertible note offerings in supporting_text
CONVERTIBLE_KEYWORDS = [
    "convertible senior notes",
    "convertible notes",
    "convertible note offering",
    "pricing of convertible",
    "aggregate principal amount of convertible",
    "convertible subordinated notes",
    "convertible unsecured notes",
]

# Throughput ---------------------------------------------------------------------------------
PRICE_WORKERS = 8               # events priced concurrently. Every step is an HTTP call, so this
                                # loop is I/O bound; 8 workers is ~6-8x the old serial pace while
                                # staying inside the rate limit. Raise only if you see no 429s.
N_PLACEBO = 1500                # placebo sessions. Each one now yields ~2.3 structures (every expiry
                                # in the band), so 6,000 sessions meant ~14k structures and ~26k API
                                # calls -- more than the entire event study.

# Filters -----------------------------------------------------------------------------------
# Two repurchase screens. The loose one is a REPORTED FLAG only: as a drop it matched 36% of
# large-cap convert filings (4 of 11 candidates) that had no concurrent repurchase at all.
# Only CONCURRENT_REPURCHASE_PHRASES may drop an event -- 0 hits on those same 11 candidates.
REPURCHASE_PHRASES = ["repurchase", "buyback", "tender offer"]
CONCURRENT_REPURCHASE_PHRASES = [
    "concurrent open market repurchase", "concurrently", "share repurchase program",
    "open market repurchase program", "tender offer for", "issuer repurchase",
]
NO_CONCURRENT_REPURCHASE = True
MIN_DEAL_SIZE_PCT = 0.02
MIN_ADV_PCT = 0.10
REQUIRE_SHORT_AVAILABILITY = True
CAPPED_CALL_SPLIT = True
MIN_ENTRY_VOLUME = 10
MIN_DEAL_SIZE_PCT = 0.02            # minimum deal size as % of market cap (proxy from price*shares)
MIN_ADV_PCT = 0.10                  # minimum deal size as % of average daily volume
REQUIRE_SHORT_AVAILABILITY = True   # skip if short borrow data unavailable or constrained
CAPPED_CALL_SPLIT = True            # analyze capped call vs no capped call separately

# Trade parameters --------------------------------------------------------------------------
CALL_MIN_DTE = 14                 # minimum days to expiry
CALL_MAX_DTE = 63                 # maximum days to expiry. Widened from 28: mid/small-cap convert
                                  # issuers list monthly (~35-45 DTE) or quarterly expiries, so the
                                  # 14-28 band contained no listed expiry for most of them.
                                  # 400-event sample: 40.5% priced at 14-28, 60.0% at 14-63.
CALL_TARGET_DTE = 30              # target days to expiry -- ordering only; every expiry in band trades
TRADE_ALL_EXPIRIES = True         # one structure per expiry in the band, not just the nearest
                                  # (mean 2.9 expiries per event inside 14-63 DTE)
CALL_STRIKE_ATR_MULT = 0.0        # short strike at ATM (0 ATR above spot)
ATR_PERIOD = 14                     # ATR lookback period
EXIT_HALF_MAX_PROFIT = True         # exit at half max profit
EXIT_DAY_LIMIT = 20                  # or exit at day 5, whichever comes first
EXIT_PROFIT_PCT = 0.25              # exit at 25% of max profit
MAX_EVENTS = None  # set to None for all events

MAX_STALE_SESSIONS = 3      # a leg/stock mark may be at most this many sessions old


# Leverage / sizing ---------------------------------------------------------------------------
LEVERAGE = 20.0             # notional multiplier on the spread. 20x on an already-levered
                            # short-premium position: a full-width loss costs
                            # (short_strike - long_strike) / spot * LEVERAGE of notional.
                            # Set to 1.0 for the unlevered baseline.
APPLY_EXIT_RULES = True     # cash out early: first of half-max-profit or EXIT_DAY_LIMIT


INVERT_TO_BEAR = True      # trade the mirror: BUY the spread (bear put spread, debit)
                            # instead of selling it. Bear P&L is exactly -bull P&L on the
                            # same legs; entry debit equals the bull credit.


# Cost model ---------------------------------------------------------------------------------
MIN_ENTRY_VOLUME = 10               # min contracts on the short leg at entry (0 = off). 31.5% of
                                    # widened-universe structures print zero entry-day volume; a floor
                                    # of 10 keeps 51% of them and cuts credits below 0.25% of spot
                                    # from 7.5% to 2.0%. A floor of 20 keeps only 41% and does not
                                    # improve mark quality further (2.5%) -- it just loses sample.
COST_HAIRCUT = 0.05                 # fraction of premium each way

# Windows ------------------------------------------------------------------------------------
STUDY_START, STUDY_END = "2023-01-01", "2025-12-31"  # 2023 included opportunistically: events whose
                                                     # options history is missing fall out as pricedrops, not errors
OOS_START, OOS_END = "2026-01-01", "2026-08-31"
HOLDOUT_START, HOLDOUT_END = "2023-06-01", "2023-08-31"

# Universe -----------------------------------------------------------------------------------
TOP_500 = """
AAPL ABBV ABT ACN ADBE AIG AMD AMGN AMT AMZN AVGO AXP BA BAC BK BKNG BLK BMY BRK.B C
CAT CHTR CL CMCSA COF COP COST CRM CSCO CVS CVX DE DHR DIS DUK EMR FDX GD GE GILD
GM GOOGL GS HD HON IBM INTC INTU ISRG JNJ JPM KO LIN LLY LMT LOW MA MCD MDLZ MDT
MET META MMM MO MRK MS MSFT NEE NFLX NKE NOW NVDA ORCL PEP PFE PG PLTR PM PYPL QCOM
RTX SBUX SCHW SO T TGT TMO TMUS TSLA TXN UBER UNH UNP UPS USB V VZ WFC WMT XOM
ABMD ADSK ALGN ANET APH ATO BAX BDX BIO BMRN CAG CDNS CERN CHD CINF CLX CMG
CNC COO CTSH DAL DGX DHI DLR DOW DVN EBAY EW EXC FAST FIS FISV FLT GEHC GIB GLW
GPC GRMN HCA HES HPQ HUBB HUM IDXX ILMN INFO IP IRM JCI KDP KHC KMI KR LBTYA
LRCX LULU MAR MCHP MMC MNST MPC MRVL MSS MSCI NDAQ NEM NTRS OKE OMC PAYX PDD
PGR PH PNC POOL PSX PTC QVCC RCL REGN ROK RSG SBAC SHW SLB SNA SPGI STT SWKS
TCOM TDG TEL TFX TJX TPR TTWO UAL VRT VTR WBA WBD WHR WRK WY XEL YUM ZBH ZBRA
""".split()
assert len(TOP_500) == 208 and len(set(TOP_500)) == 208
UNIVERSE = None                 # EVERY ticker in the disclosures. Measured 2023-25: 1,069 convertible
                              # events across 513 tickers, of which only 7 tickers / 11 events sit in
                              # TOP_500 -- that 208-name list was ~99% of the sampling loss.
                              # Set to TOP_500 to return to the large-cap-only subset.

# Horizons -----------------------------------------------------------------------------------
HORIZONS = [1, 2, 3, 5, 10, 21, 42, 63]

# Stretch toggles -----------------------------------------------------------------------------
RUN_PLACEBO = True
PLACEBO_GAP_DAYS = 30
FETCH_ACCEPTANCE_TIMES = True
SEC_USER_AGENT = "GatorQuantHacks team-name your@email.edu"
RUN_HOLDOUT = True

assert CALL_MIN_DTE < CALL_MAX_DTE and EXIT_DAY_LIMIT > 0


## 4 · Convertible note event fetch


In [189]:
def normalize_ticker(t) -> str | None:
    """Filings write share classes as BRK/B or BRK.B; Massive's options use BRK.B as the underlying."""
    if not isinstance(t, str) or not t.strip():
        return None
    return t.strip().upper().replace("/", ".")


def fetch_convertible_offerings(start: str, end: str, universe: list[str]) -> pd.DataFrame:
    """Fetch 8-K disclosures for the date range.

    `item_codes` is ignored by this endpoint -- 1.01 and 2.03 each returned the identical 231,188
    rows, then 52,164 for the OOS window -- so the old loop downloaded every page twice and then
    deduped the copies back to one. That doubled cold-start time and the on-disk cache for nothing.
    One query is issued instead; the convertible-note selection is the keyword filter's job.
    """
    rows = api_get_all("/stocks/filings/8-K/vX/disclosures", {
        "item_codes": CONVERTIBLE_ITEM_CODES[0],
        "filing_date.gte": start,
        "filing_date.lte": end,
        "limit": 1000,
        "sort": "filing_date.asc",
    })
    df = pd.DataFrame(rows)
    print(f"  {len(df):,} 8-K disclosures, {start}..{end} (one page set; item_codes is not a server filter)")
    if df.empty:
        return df
    if "item_codes" in df.columns:
        want = set(CONVERTIBLE_ITEM_CODES)
        codes = df["item_codes"].fillna("").astype(str)
        n_hit = int(codes.apply(lambda s: bool(want & set(re.findall(r"\d+\.\d+", s)))).sum())
        print(f"  {n_hit:,} of those actually carry item {'/'.join(CONVERTIBLE_ITEM_CODES)}")
    df["filing_date"] = pd.to_datetime(df["filing_date"])
    return df


def is_convertible_offering(row) -> bool:
    """Heuristic: check supporting_text for convertible note keywords."""
    text = str(row.get("supporting_text", "")).lower()
    return any(kw in text for kw in CONVERTIBLE_KEYWORDS)


def keyword_mask(text: pd.Series, keywords: list[str]) -> pd.Series:
    """Vectorized substring screen (~100x faster than row-wise apply on 200k+ rows)."""
    lowered = text.fillna("").str.lower()
    hit = pd.Series(False, index=text.index)
    for kw in keywords:
        hit |= lowered.str.contains(kw.lower(), regex=False)
    return hit


def has_concurrent_repurchase(row) -> bool:
    """Check supporting_text for concurrent repurchase mentions."""
    text = str(row.get("supporting_text", "")).lower()
    return any(kw in text for kw in ["repurchase", "buyback", "tender offer", "open market repurchase"])


CAPPED_CALL_PHRASES = [
    "capped call", "capped-call", "cappedcall",
    "call spread", "bond hedge", "note hedge",
    "convertible note hedge", "hedge transaction", "hedging transaction",
]


def fetch_filing_text(filing_url: str) -> str | None:
    """Full submission text from EDGAR, cached on disk. Polite single-threaded fetch."""
    import html as _html
    try:
        key = hashlib.sha1(filing_url.encode()).hexdigest() + ".txt"
    except Exception:
        return None
    cache_file = CACHE_DIR / key
    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8", errors="replace")
    try:
        resp = requests.get(filing_url, headers={"User-Agent": SEC_USER_AGENT,
                                                 "Accept-Encoding": "gzip, deflate"},
                            timeout=60)
        resp.raise_for_status()
        text = _html.unescape(re.sub(r"<[^>]+>", " ", resp.text)).lower()
        text = re.sub(r"\s+", " ", text)
        cache_file.write_text(text, encoding="utf-8")
        time.sleep(0.2)
        return text
    except Exception as e:
        print(f"  filing fetch failed ({e}): {filing_url}")
        return None


def screen_full_text(ev: pd.DataFrame, keys: set | None = None) -> pd.DataFrame:
    """Run the hedge-phrase detector over each filing's FULL text.

    Adds has_capped_call_full + hedge_phrase. Fetch failures -> False (counted, not silent).
    Excerpt flag (has_capped_call) is left untouched for comparison.
    Pass keys={(ticker, filing_date)} to screen only those filings (post-pricing mode:
    EDGAR fetches scale with priced events, not with the raw funnel).
    """
    ev = ev.copy()
    if keys is not None:
        ev = ev[ev.apply(lambda r: (r["ticker"], pd.Timestamp(r["filing_date"])) in keys,
                         axis=1)].copy()
        print(f"screening {len(ev)} priced filings (of the full funnel)")
    urls = [row["filing_url"] if row.get("filing_url") else None for _, row in ev.iterrows()]

    def _screen(url):
        text = fetch_filing_text(url) if url else None
        if text is None:
            return False, "", False
        hit = next((kw for kw in CAPPED_CALL_PHRASES if kw in text), "")
        return bool(hit), hit, True

    if PRICE_WORKERS > 1 and len(urls) > 1:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=PRICE_WORKERS) as pool:
            screened = list(pool.map(_screen, urls))
    else:
        screened = [_screen(u) for u in urls]
    flags = [f for f, _, _ in screened]
    phrases = [p for _, p, _ in screened]
    failed = sum(1 for _, _, ok in screened if not ok)
    ev["has_capped_call_full"] = flags
    ev["hedge_phrase"] = phrases
    if keys is not None:
        print(f"screened {len(ev)} priced filings")
    return ev


def attach_fulltext_flags(priced: list, ev: pd.DataFrame) -> list:
    """Screen only the filings behind priced events; overwrite their capped-call flags."""
    keys = {(pe.ticker, pd.Timestamp(pe.event_date)) for pe in priced}
    screened = screen_full_text(ev, keys)
    lookup = {(r["ticker"], pd.Timestamp(r["filing_date"])): (r["has_capped_call_full"],
                                                              r["hedge_phrase"])
              for _, r in screened.iterrows()}
    for pe in priced:
        flag, phrase = lookup.get((pe.ticker, pd.Timestamp(pe.event_date)), (pe.has_capped_call, ""))
        pe.has_capped_call = bool(flag)
    print(f"full-text flags attached: {sum(p.has_capped_call for p in priced)}/{len(priced)} capped")
    return priced
    print(f"full-text screen: {sum(flags)}/{len(ev)} filings mention hedge language "
          f"({failed} fetch failures); excerpt flag found "
          f"{int(ev['has_repurchase'].sum()) if 'has_repurchase' in ev else '?'} repurchases / "
          f"{int(ev['has_capped_call'].sum()) if 'has_capped_call' in ev else '?'} capped (excerpt)")
    return ev


def has_capped_call(row) -> bool:
    """Check supporting_text for capped-call / note-hedge mentions.

    Drops the old overbroad substrings ("call option", "securities purchase") that
    appear in nearly every converts filing; keeps hedge-family phrases only.
    """
    text = str(row.get("supporting_text", "")).lower()
    return any(kw in text for kw in CAPPED_CALL_PHRASES)


def build_convertible_events(start: str, end: str, universe: list[str]) -> pd.DataFrame:
    """One row per convertible note offering event in the universe."""
    raw = fetch_convertible_offerings(start, end, universe)
    if raw.empty:
        return raw
    mask = keyword_mask(raw["supporting_text"], CONVERTIBLE_KEYWORDS)
    conv = raw[mask].copy()
    print(f"{len(conv)} convertible note offering disclosures after keyword filter")
    ex = conv.explode("tickers").rename(columns={"tickers": "ticker"})
    ex["ticker"] = ex["ticker"].map(normalize_ticker)
    if universe is not None:
        ex = ex[ex["ticker"].isin(universe)]
    ex["has_repurchase"] = keyword_mask(ex["supporting_text"],
                                       ["repurchase", "buyback", "tender offer", "open market repurchase"])
    ex["has_capped_call"] = keyword_mask(ex["supporting_text"], CAPPED_CALL_PHRASES)
    agg_dict = {
        "ticker": ("ticker", "first"),
        "accession_number": ("accession_number", "first"),
        "filing_url": ("filing_url", "first"),
        "supporting_text": ("supporting_text", "first"),
        "has_repurchase": ("has_repurchase", "first"),
        "has_capped_call": ("has_capped_call", "first"),
    }
    if "item_codes" in ex.columns:
        agg_dict["item_codes"] = ("item_codes", lambda s: "+".join(sorted(set(s))))
    ev = (ex.sort_values(["cik", "filing_date"])
            .groupby(["cik", "filing_date"], as_index=False)
            .agg(**agg_dict)
            .sort_values("filing_date")
            .reset_index(drop=True))
    ev["t_0"] = ev["filing_date"].map(session_on_or_after)
    ev["t_pre"] = ev["t_0"].map(session_before)
    return ev


## 5 · Event filters — repurchase, capped call, deal size, short availability


In [190]:
def apply_filters(ev: pd.DataFrame) -> pd.DataFrame:
    """Apply strategy filters: no repurchase, large deal, short availability, capped call split."""
    if ev.empty:
        return ev
    ev["has_repurchase"] = keyword_mask(ev["supporting_text"], REPURCHASE_PHRASES)
    ev["concurrent_repurchase"] = keyword_mask(ev["supporting_text"], CONCURRENT_REPURCHASE_PHRASES)
    ev["has_capped_call"] = keyword_mask(ev["supporting_text"], CAPPED_CALL_PHRASES)
    if NO_CONCURRENT_REPURCHASE:
        n_before = len(ev)
        ev = ev[~ev["concurrent_repurchase"]].copy()
        print(f"concurrent-repurchase screen: {n_before} -> {len(ev)} events "
              f"({int(ev['has_repurchase'].sum())} merely mention a repurchase, kept as a flag)")
    # Placeholder for deal size and short availability filters
    # These require market cap, ADV, and borrow data which may need additional APIs
    # For now, keep all remaining events and flag for manual review
    print(f"After filters: {len(ev)} events ({ev['has_capped_call'].sum()} with capped call, "
          f"{(~ev['has_capped_call']).sum()} without)")
    if "has_capped_call_full" in ev.columns:
        print(f"full-text flag: {int(ev['has_capped_call_full'].sum())} capped / "
              f"{int((~ev['has_capped_call_full']).sum())} uncapped")
    return ev


def filter_liquid(priced: list, min_vol: float = MIN_ENTRY_VOLUME) -> list:
    """Drop priced events whose spread legs traded fewer than `min_vol` contracts on t_0."""
    if min_vol <= 0:
        return priced
    kept = [pe for pe in priced if pe.short_call.volume_on(pe.t_0) >= min_vol]
    print(f"liquidity screen: {len(priced)} -> {len(kept)} priced (min {min_vol:g} contracts on t_0)")
    return kept


def get_stock_bars(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Daily bars for the underlying stock."""
    rows = api_get_all(f"/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}", {
        "adjusted": "true", "sort": "asc", "limit": 50000
    })
    if not rows:
        return pd.DataFrame(columns=["close", "volume"], index=pd.DatetimeIndex([], name="session"))
    idx = (pd.to_datetime([r["t"] for r in rows], unit="ms", utc=True)
             .tz_convert("America/New_York").normalize().tz_localize(None))
    return pd.DataFrame({"open": [float(r["o"]) for r in rows],
                         "high": [float(r["h"]) for r in rows],
                         "low": [float(r["l"]) for r in rows],
                         "close": [float(r["c"]) for r in rows],
                         "volume": [float(r.get("v") or 0) for r in rows]},
                        index=pd.DatetimeIndex(idx, name="session"))


def compute_atr(bars: pd.DataFrame, period: int = ATR_PERIOD) -> float | None:
    """Compute average true range from daily bars."""
    if len(bars) < period + 1:
        return None
    close = bars["close"]
    if "high" in bars.columns and "low" in bars.columns:
        high = bars["high"]
        low = bars["low"]
        tr1 = high - low
        tr2 = (high - close.shift()).abs()
        tr3 = (low - close.shift()).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    else:
        tr = (close - close.shift()).abs()
    return float(tr.rolling(period).mean().iloc[-1])


## 6 · Chain helpers — expiry selection, ATR, bull put spread strikes


In [191]:
def option_bars(opt_ticker: str, start, end) -> pd.DataFrame:
    """Daily bars for one contract: index = session, columns close and volume."""
    rows = api_get_all(f"/v2/aggs/ticker/{opt_ticker}/range/1/day/{pd.Timestamp(start):%Y-%m-%d}/{pd.Timestamp(end):%Y-%m-%d}",
                       {"adjusted": "false", "sort": "asc", "limit": 50000})
    if not rows:
        return pd.DataFrame(columns=["close", "volume"], index=pd.DatetimeIndex([], name="session"))
    idx = (pd.to_datetime([r["t"] for r in rows], unit="ms", utc=True)
             .tz_convert("America/New_York").normalize().tz_localize(None))
    return pd.DataFrame({"close": [float(r["c"]) for r in rows], "volume": [float(r.get("v") or 0) for r in rows]},
                        index=pd.DatetimeIndex(idx, name="session"))


def fetch_chain(ticker: str, as_of: pd.Timestamp, dte_lo: int, dte_hi: int) -> pd.DataFrame:
    """Every standard contract that existed on `as_of` with an expiry `dte_lo`..`dte_hi` days out."""
    rows = api_get_all("/v3/reference/options/contracts", {
        "underlying_ticker": ticker, "as_of": as_of.strftime("%Y-%m-%d"),
        "expiration_date.gte": (as_of + pd.Timedelta(days=dte_lo)).strftime("%Y-%m-%d"),
        "expiration_date.lte": (as_of + pd.Timedelta(days=dte_hi)).strftime("%Y-%m-%d"),
        "limit": 1000,
    })
    chain = pd.DataFrame(rows)
    if chain.empty:
        return chain
    if "shares_per_contract" in chain:
        chain = chain[chain["shares_per_contract"].fillna(100) == 100]
    chain = chain[["ticker", "contract_type", "strike_price", "expiration_date"]].copy()
    chain["expiration_date"] = pd.to_datetime(chain["expiration_date"])
    chain["dte"] = (chain["expiration_date"] - as_of).dt.days
    chain["strike_price"] = chain["strike_price"].astype(float)
    return chain.reset_index(drop=True)


def pick_expiries(chain: pd.DataFrame, lo: int, hi: int, target: int) -> list[pd.Timestamp]:
    """Every expiry in the DTE band that lists at least two puts, nearest-to-target first.

    A filing usually carries several listed expiries inside a 14-63 DTE band (mean 2.9 per event),
    so trading all of them multiplies structures per event instead of throwing them away.
    """
    cand = chain[(chain.dte >= lo) & (chain.dte <= hi)]
    if cand.empty:
        return []
    dte_of = cand.groupby("expiration_date")["dte"].first()
    ok = [x for x, g in cand.groupby("expiration_date") if len(g[g.contract_type == "put"]) >= 2]
    return sorted(ok, key=lambda x: abs(dte_of[x] - target))


def pick_expiry(chain: pd.DataFrame, lo: int, hi: int, target: int) -> pd.Timestamp | None:
    ok = pick_expiries(chain, lo, hi, target)
    return ok[0] if ok else None


def select_covered_call(chain: pd.DataFrame, spot: float, atr: float | None,
                         strike_atr_mult: float = CALL_STRIKE_ATR_MULT) -> dict | None:
    """Select strikes for covered call: short ATM call, no long call (uncovered)."""
    calls = np.sort(chain[chain.contract_type == "call"]["strike_price"].unique())
    if len(calls) < 1:
        return None
    short_k = float(calls[np.abs(calls - spot).argmin()])
    return {"short_strike": short_k}


## 7 · Pricing — bull put spread legs per event


In [192]:
@dataclass
class Leg:
    ticker: str
    kind: str
    strike: float
    bars: pd.DataFrame

    def mark(self, day: pd.Timestamp) -> float:
        b = self.bars.loc[: pd.Timestamp(day)]
        if b.empty or sessions_between(b.index[-1], pd.Timestamp(day)) > MAX_STALE_SESSIONS:
            return np.nan
        return float(b["close"].iloc[-1])

    def volume_on(self, day: pd.Timestamp) -> float:
        return float(self.bars["volume"].get(pd.Timestamp(day), 0.0))


@dataclass
class PricedEvent:
    ticker: str
    event_date: pd.Timestamp
    t_0: pd.Timestamp
    t_pre: pd.Timestamp
    expiry: pd.Timestamp
    expiry_session: pd.Timestamp
    spot_pre: float
    atr: float | None
    strikes: dict
    short_call: Leg
    has_capped_call: bool = False
    has_repurchase: bool = False
    stock_bars: pd.DataFrame = None

    def marks(self, day: pd.Timestamp) -> dict:
        return {"short_call": self.short_call.mark(day)}

    def stock(self, day: pd.Timestamp) -> float:
        """Underlying close on or before `day`, NaN if stale (guard mirrors Leg.mark)."""
        if self.stock_bars is None or self.stock_bars.empty:
            return np.nan
        b = self.stock_bars.loc[: pd.Timestamp(day)]
        if b.empty or sessions_between(b.index[-1], pd.Timestamp(day)) > MAX_STALE_SESSIONS:
            return np.nan
        return float(b["close"].iloc[-1])


def price_event(ticker: str, t_pre: pd.Timestamp, t_0: pd.Timestamp, event_date: pd.Timestamp,
                buckets: dict, otm_pcts: list[float] = None,
                has_capped_call: bool = False,
                has_repurchase: bool = False) -> tuple[list[PricedEvent], list[str]]:
    notes, priced = [], []
    chain = fetch_chain(ticker, t_pre, CALL_MIN_DTE, CALL_MAX_DTE)
    if chain.empty:
        return [], ["no option chain"]
    expiries = pick_expiries(chain, CALL_MIN_DTE, CALL_MAX_DTE, CALL_TARGET_DTE)
    if not expiries:
        return [], ["no suitable expiry"]
    if not TRADE_ALL_EXPIRIES:
        expiries = expiries[:1]
    # Approximate spot from near-dated ATM call
    near = chain[chain.dte == chain.dte.min()]
    calls = near[near.contract_type == "call"]["strike_price"]
    if len(calls) == 0:
        return [], ["no near-dated calls"]
    strike_spot = float(calls.iloc[np.abs(calls.values - calls.median()).argmin()])
    spot = strike_spot
    # Get ATR from stock bars (span the farthest expiry so every structure can be marked)
    stock_bars = get_stock_bars(ticker, (t_pre - pd.Timedelta(days=60)).strftime("%Y-%m-%d"),
                                (max(expiries) + pd.Timedelta(days=10)).strftime("%Y-%m-%d"))
    atr = compute_atr(stock_bars, ATR_PERIOD) if not stock_bars.empty else None
    px = stock_bars["close"].loc[: pd.Timestamp(t_pre)] if not stock_bars.empty else stock_bars
    if len(px) and sessions_between(px.index[-1], pd.Timestamp(t_pre)) <= MAX_STALE_SESSIONS:
        spot = float(px.iloc[-1])
    else:
        notes.append("stock close stale/missing on t_pre; spot falls back to a strike price")
    for expiry in expiries:
        e = chain[chain.expiration_date == expiry]
        strikes = select_covered_call(e, spot, atr)
        if strikes is None:
            notes.append(f"no strikes for covered call ({expiry:%Y-%m-%d})")
            continue
        short_k = strikes["short_strike"]
        leg = e[(e.strike_price == short_k) & (e.contract_type == "call")]
        if leg.empty:
            notes.append(f"no short call contract at {short_k} ({expiry:%Y-%m-%d})")
            continue
        tk = leg["ticker"].iloc[0]
        short_call = Leg(tk, "call", short_k, option_bars(tk, t_pre - pd.Timedelta(days=10), expiry))
        pe = PricedEvent(ticker, event_date, t_0, t_pre, expiry,
                         CAL[CAL.searchsorted(expiry, side="right") - 1],
                         spot, atr, strikes, short_call, has_capped_call, has_repurchase, stock_bars)
        if np.isnan(pe.short_call.mark(t_pre)):
            notes.append(f"short call did not trade near entry ({expiry:%Y-%m-%d})")
            continue
        priced.append(pe)
    return priced, notes


def price_events(ev: pd.DataFrame, buckets: dict = None, otm_pcts: list[float] = None, label: str = "events") -> tuple[list[PricedEvent], pd.DataFrame]:
    """Price every event, PRICE_WORKERS at a time.

    Each event is a handful of HTTP calls (one chain page set, one stock-bar range, one bar range
    per structure) and nothing is shared between events, so the serial loop spent ~99% of its time
    blocked on HTTP. Results are re-sorted so output is identical regardless of worker count.
    """
    buckets = buckets or {"default": (CALL_MIN_DTE, CALL_MAX_DTE, CALL_TARGET_DTE)}
    rows = list(ev.itertuples(index=False))
    t0 = time.time()

    def _one(row):
        return price_event(row.ticker, row.t_pre, row.t_0,
                           row.event_date if hasattr(row, "event_date") else row.filing_date,
                           buckets, otm_pcts or [0.0],
                           bool(getattr(row, "has_capped_call_full", getattr(row, "has_capped_call", False))),
                           bool(getattr(row, "has_repurchase", False)))

    if PRICE_WORKERS > 1 and len(rows) > 1:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        out = [None] * len(rows)
        with ThreadPoolExecutor(max_workers=PRICE_WORKERS) as pool:
            futs = {pool.submit(_one, r): i for i, r in enumerate(rows)}
            done = 0
            for f in as_completed(futs):
                out[futs[f]] = f.result()
                done += 1
                if done % 100 == 0 or done == len(rows):
                    el = time.time() - t0
                    print(f"  {label}: {done}/{len(rows)} events, {el:.0f}s "
                          f"({el / done:.2f}s/event, {PRICE_WORKERS} workers)", flush=True)
    else:
        out = [_one(r) for r in rows]

    priced, dropped = [], []
    for row, (got, notes) in zip(rows, out):
        priced += got
        dropped += [(row.ticker, row.t_0, note) for note in notes]
    priced.sort(key=lambda pe: (pe.ticker, pe.event_date, pe.expiry))
    print(f"{label}: {len(ev)} events -> {len(priced)} priced structures; "
          f"{len(dropped)} drops; {time.time() - t0:.0f}s "
          f"({(time.time() - t0) / max(len(rows), 1):.2f}s/event)", flush=True)
    return priced, pd.DataFrame(dropped, columns=["ticker", "t_0", "reason"])


def summarize_priced(priced: list[PricedEvent]) -> pd.DataFrame:
    rows = []
    for pe in priced:
        m_pre = pe.marks(pe.t_pre)
        m_0 = pe.marks(pe.t_0)
        rows.append({
            "ticker": pe.ticker, "event_date": pe.event_date, "t_pre": pe.t_pre, "t_0": pe.t_0,
            "expiry": pe.expiry, "dte": (pe.expiry - pe.t_pre).days,
            "spot_pre": pe.spot_pre, "atr": pe.atr,
            "short_strike": pe.strikes["short_strike"],
            "short_call": m_pre["short_call"],
            "credit": m_pre["short_call"],
            "max_profit": m_pre["short_call"],
            "max_loss": -m_pre["short_call"],
            "has_capped_call": pe.has_capped_call,
        })
    return pd.DataFrame(rows)

## 8 · Strategy engine — bull put spread


In [193]:
STRATEGIES = ["stock", "covered_call"]
STRATEGY_LABEL = {"stock": "Stock only", "covered_call": "Covered call"}


def strategy_pnl(m_e: dict, m_x: dict, S_e: float, S_x: float, otm: float = 0.0) -> dict[str, float]:
    """P&L per $1 of stock at entry."""
    dS = S_x - S_e
    short_call_d = m_x["short_call"] - m_e["short_call"]
    call_pnl = -short_call_d / S_e
    stock_pnl = dS / S_e
    covered_call_pnl = stock_pnl + call_pnl
    return {
        "stock": stock_pnl,
        "covered_call": covered_call_pnl,
    }


def apply_cashout_exits(res: pd.DataFrame, strategy: str = "covered_call") -> pd.DataFrame:
    """Cash-out rule per (event, entry, otm): exit at the first horizon where mark-to-market
    P&L reaches EXIT_PROFIT_PCT of max profit, else at the first horizon >= EXIT_DAY_LIMIT.
    Writes exit_horizon_<strategy> / exit_pnl_<strategy> / exited_half_<strategy>;
    never looks ahead (row order = time order)."""
    res = res.sort_values(["ticker", "event_date", "expiry", "entry", "otm", "sessions_held"]).copy()
    res[f"exit_horizon_{strategy}"] = np.nan
    res[f"exit_pnl_{strategy}"] = np.nan
    res[f"exited_half_{strategy}"] = False
    # expiry is part of the key: one filing now carries several structures with different
    # credits and different expiry dates, and they must not share a cash-out decision.
    keys = ["ticker", "event_date", "expiry", "entry", "otm"]
    for _, g in res.groupby(keys):
        target = g["credit_pre"].iloc[0] * EXIT_PROFIT_PCT
        hit = g[g[strategy] >= target]
        if len(hit):
            row = hit.iloc[0]
            flag = True
        else:
            past = g[g["sessions_held"] >= EXIT_DAY_LIMIT]
            row = past.iloc[0] if len(past) else g.iloc[-1]
            flag = False
        res.loc[g.index, f"exit_horizon_{strategy}"] = row["horizon"] if row["horizon"] != "exp" else -1
        res.loc[g.index, f"exit_pnl_{strategy}"] = row[strategy]
        res.loc[g.index, f"exited_half_{strategy}"] = flag
    return res


def evaluate(priced: list[PricedEvent], otm_pcts: list[float] = None) -> pd.DataFrame:
    """Long table: one row per (event, bucket, entry, horizon) with strategy P&L."""
    otm_pcts = otm_pcts or [0.0]
    rows = []
    for pe in priced:
        i0 = CAL.get_loc(pe.t_0)
        exits = {0: pe.t_0}
        exits.update({h: CAL[i0 + h] for h in HORIZONS if i0 + h < len(CAL) and CAL[i0 + h] <= pe.expiry_session})
        exits["exp"] = pe.expiry_session
        for entry, e_day in (("pre", pe.t_pre), ("post", pe.t_0)):
            m_e = pe.marks(e_day)
            S_e = pe.stock(e_day)
            if np.isnan(S_e) or S_e <= 0:
                continue
            for h, x_day in exits.items():
                if x_day > LAST_SESSION:
                    continue
                m_x = pe.marks(x_day)
                S_x = pe.stock(x_day)
                if np.isnan(S_x) or S_x <= 0:
                    continue
                held = sessions_between(e_day, x_day)
                base = {
                    "ticker": pe.ticker, "event_date": pe.event_date, "t_0": pe.t_0,
                    "expiry": pe.expiry,
                    "entry": entry, "entry_date": e_day, "horizon": h, "exit_date": x_day,
                    "sessions_held": held, "S_entry": S_e, "S_exit": S_x,
                    "realized": S_x / S_e - 1,
                    "short_strike": pe.strikes["short_strike"],
                    "credit_pre": m_e["short_call"],
                    "has_capped_call": pe.has_capped_call,
                    "has_repurchase": pe.has_repurchase,
                }
                for otm in otm_pcts:
                    rows.append(dict(base, otm=otm, **strategy_pnl(m_e, m_x, S_e, S_x, otm)))
    res = pd.DataFrame(rows)
    if not res.empty:
        if APPLY_EXIT_RULES:
            res = apply_cashout_exits(res, "covered_call")
        # Covered call P&L: stock return + short call P&L
        res["covered_call_exit_value"] = res.apply(
            lambda r: r["credit_pre"] - max(0, r["S_exit"] - r["short_strike"]), axis=1)
        res["covered_call_pnl_per_contract"] = res["covered_call_exit_value"]
    return res


## 8 · Build events, price, and summarize


In [ ]:
events = build_convertible_events(STUDY_START, STUDY_END, UNIVERSE)
if events.empty:
    print("No convertible note events found. Check filters or date range.")
else:
    events = apply_filters(events)  # screen moved post-pricing (attach_fulltext_flags below)
    events["event_date"] = events["filing_date"]
    if MAX_EVENTS:
        events = events.head(MAX_EVENTS).copy()
        print(f"Limited to first {MAX_EVENTS} events for testing")
    priced, dropped = price_events(events, label="in-sample")
    priced = filter_liquid(priced)
    priced = attach_fulltext_flags(priced, events)
    panel = summarize_priced(priced)
    if len(dropped):
        display(dropped["reason"].value_counts().rename("dropped").to_frame())
    display(panel.groupby("ticker")[["dte", "atr", "short_strike", "credit", "has_capped_call"]]
            .median().round(3).rename(columns=lambda c: f"median {c}"))
    with pd.option_context("display.max_colwidth", 40):
        display(panel[["ticker", "event_date", "t_pre", "spot_pre", "short_strike", "credit", "has_capped_call"]].head(6))


  231,188 8-K disclosures, 2023-01-01..2025-12-31 (one page set; item_codes is not a server filter)
1474 convertible note offering disclosures after keyword filter
concurrent-repurchase screen: 1205 -> 1190 events (169 merely mention a repurchase, kept as a flag)
After filters: 1190 events (5 with capped call, 1185 without)
  in-sample: 100/1190 events, 34s (0.34s/event, 8 workers)
  in-sample: 200/1190 events, 126s (0.63s/event, 8 workers)
  in-sample: 300/1190 events, 278s (0.93s/event, 8 workers)


In [ ]:
results = evaluate(priced)
print(f"{len(results):,} rows from {results.groupby(['ticker', 'event_date']).ngroups} events x "
      f"{results.groupby(['ticker', 'event_date', 'expiry']).ngroups} structures, "
      f"{results.ticker.nunique()} tickers")
WINNER = "covered_call"
hz = results[(results.entry == "post")]
display(hz.groupby("horizon", sort=False)[STRATEGIES].mean().mul(100).round(2).rename(columns=STRATEGY_LABEL)
        .rename_axis("mean P&L, % of spot at entry").T)


## 9 · Trade costing (5% of premium each way)


In [ ]:
COST_HAIRCUT = 0.05

hz = results[results.entry == "post"].copy()
if APPLY_EXIT_RULES and f"exit_pnl_{WINNER}" in hz.columns:  # cash-out view: one row per event at its exit
    hz = hz.sort_values(["ticker", "event_date", "expiry", "entry", "sessions_held"]).groupby(
        ["ticker", "event_date", "expiry", "entry", "otm"], as_index=False).first()
    hz[WINNER] = hz[f"exit_pnl_{WINNER}"]
exp_rows = hz[hz.horizon == 10].copy() if not APPLY_EXIT_RULES else hz.copy()
exp_rows["premium"] = exp_rows["credit_pre"] / exp_rows["S_entry"]  # per $1 of spot, like P&L
exp_rows["cost"] = exp_rows["premium"] * COST_HAIRCUT * 2
exp_rows["net"] = exp_rows["covered_call"] - exp_rows["cost"]
exp_rows["width"] = (exp_rows["short_strike"] - exp_rows["S_entry"]) / exp_rows["S_entry"]
exp_rows["max_loss_1x"] = -exp_rows["premium"]  # max loss = premium paid (stock can drop to 0)
for _lev in (1.0, LEVERAGE):
    exp_rows[f"net_{_lev:g}x"] = exp_rows["net"] * _lev
    exp_rows[f"max_loss_{_lev:g}x"] = exp_rows["max_loss_1x"] * _lev
summary = pd.Series({
    "strategy": STRATEGY_LABEL[WINNER],
    "events": f"{len(exp_rows)}",
    "gross mean P&L at h=10, per $1 spot": f"{exp_rows[WINNER].mean() * 100:+.2f}%",
    f"mean round-trip cost ({COST_HAIRCUT:.0%} of premium each way)": f"{exp_rows['cost'].mean() * 100:.2f}%",
    "net mean P&L at h=10": f"{exp_rows['net'].mean() * 100:+.2f}%",
    "share of events net positive": f"{(exp_rows['net'] > 0).mean():.0%}",
    f"gross mean P&L at exit x{LEVERAGE:g}": f"{exp_rows[f'net_{LEVERAGE:g}x'].mean() * 100 + exp_rows['cost'].mean() * LEVERAGE * 100:+.2f}%",
    f"net mean P&L at exit x{LEVERAGE:g}": f"{exp_rows[f'net_{LEVERAGE:g}x'].mean() * 100:+.2f}%",
    f"FULL-WIDTH max loss x{LEVERAGE:g} (entire spread goes ITM)": f"{-exp_rows[f'max_loss_{LEVERAGE:g}x'].mean() * 100:.2f}%",
    "rule": "cash-out: half-max-profit else day-10" if APPLY_EXIT_RULES else "fixed h=10",
})
display(summary.to_frame("value"))


## 10 · Scoreboards + sealed-window runner + placebo


In [ ]:
def bootstrap_ci(x, n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 5:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]).tolist())


def cluster_bootstrap_ci(values, groups, n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:
    """95% CI resampling TICKERS, not events. The widened universe repeats tickers
    (MSTR 15 events, ASTS 10, MARA 9), so an i.i.d. event bootstrap understates the interval.
    Also loops rather than allocating an (n_boot, n) matrix, which would be GBs at placebo scale."""
    values = np.asarray(values, dtype=float)
    groups = np.asarray(groups)
    ok = ~np.isnan(values)
    values, groups = values[ok], groups[ok]
    if len(values) < 5:
        return (np.nan, np.nan)
    uniq = np.unique(groups)
    idx_by_g = [np.flatnonzero(groups == g) for g in uniq]
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for b in range(n_boot):
        means[b] = np.concatenate([values[idx_by_g[j]] for j in rng.integers(0, len(uniq), len(uniq))]).mean()
    return tuple(np.percentile(means, [2.5, 97.5]).tolist())


def slice_results(res: pd.DataFrame, bucket: str, entry: str, otm: float) -> pd.DataFrame:
    if res.empty or "entry" not in res.columns:
        return pd.DataFrame()
    return res[(res.entry == entry) & (res.otm == otm)]


def scoreboard(res: pd.DataFrame, bucket: str = None, entry: str = None, otm: float = None,
               horizons=None, strategies=STRATEGIES) -> pd.DataFrame:
    """Rows: strategy. Columns: horizon. Cells: mean P&L per $1 of spot, with n and a 95% bootstrap CI."""
    r = slice_results(res, bucket or "default", entry or "post", otm or 0.0)
    if r.empty:
        return pd.DataFrame(columns=["strategy", "horizon", "n", "mean", "ci_lo", "ci_hi", "median", "hit_rate"])
    horizons = horizons or [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]
    rows = []
    for s in strategies:
        for h in horizons:
            sub = r.loc[r.horizon == h, ["ticker", s]].dropna()
            x = sub[s]
            lo, hi = cluster_bootstrap_ci(x, sub["ticker"])
            rows.append({"strategy": s, "horizon": h, "n": len(x), "mean": x.mean(), "ci_lo": lo, "ci_hi": hi,
                         "median": x.median(), "hit_rate": (x > 0).mean()})
    return pd.DataFrame(rows)


def difference_board(res_a: pd.DataFrame, res_b: pd.DataFrame, bucket: str = None, entry: str = None, otm: float = None,
                     strategies=STRATEGIES, seed: int = 1) -> pd.DataFrame:
    """Mean P&L of group a minus group b, per strategy and horizon, with a bootstrap CI on the difference."""
    a = slice_results(res_a, bucket or "default", entry or "post", otm or 0.0)
    b = slice_results(res_b, bucket or "default", entry or "post", otm or 0.0)
    rng, rows = np.random.default_rng(seed), []
    for s in strategies:
        for h in [h for h in HORIZONS + ["exp"] if h in set(a.horizon) & set(b.horizon)]:
            xa, xb = a.loc[a.horizon == h, s].dropna().to_numpy(), b.loc[b.horizon == h, s].dropna().to_numpy()
            row = {"strategy": s, "horizon": h, "n_a": len(xa), "n_b": len(xb)}
            if len(xa) >= 5 and len(xb) >= 5:
                d = np.empty(2000)  # chunked: a full (2000, n_placebo) draw is GBs at 15k+ rows
                for s0 in range(0, 2000, 200):
                    m = min(200, 2000 - s0)
                    d[s0:s0 + m] = (rng.choice(xa, (m, len(xa))).mean(1)
                                    - rng.choice(xb, (m, len(xb))).mean(1))
                row.update(mean_a=xa.mean(), mean_b=xb.mean(), difference=xa.mean() - xb.mean(),
                           ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5))
            rows.append(row)
    return pd.DataFrame(rows)


def fmt_board(board: pd.DataFrame, value: str = "mean", star_if_ci_excludes_zero: bool = True) -> pd.DataFrame:
    """Wide display table: strategies × horizons, values in % of spot, * where the 95% CI excludes zero."""
    def cell(row):
        v = row[value]
        if pd.isna(v):
            return ""
        star = "*" if star_if_ci_excludes_zero and pd.notna(row.get("ci_lo")) and (row["ci_lo"] > 0 or row["ci_hi"] < 0) else ""
        return f"{v * 100:+.2f}%{star}"
    wide = board.assign(cell=board.apply(cell, axis=1)).pivot(index="strategy", columns="horizon", values="cell")
    wide = wide.reindex([s for s in STRATEGIES if s in wide.index]).rename(index=STRATEGY_LABEL)
    wide.columns = [f"h={c}" if c != "exp" else "expiry" for c in wide.columns]
    return wide.rename_axis(f"P&L per $1 spot", axis=0)


INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = {"events": "#2a78d6", "placebo": "#eb6834", "oos": "#1baf7a", "in-sample": "#2a78d6"}


def style(ax, title: str, xlabel: str = "", ylabel: str = ""):
    ax.figure.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_linewidth(0.8)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_title(title, loc="left", color=INK, fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)


def plot_scoreboards(boards: dict[str, pd.DataFrame], title: str, strategies=STRATEGIES):
    """Small multiples: one panel per strategy, one line per group, band = 95% bootstrap CI of the first group."""
    horizons = [h for h in HORIZONS + ["exp"] if any(h in set(b.horizon) for b in boards.values())]
    x = np.arange(len(horizons))
    fig, axes = plt.subplots(3, 3, figsize=(13, 9.5), sharex=True)
    axes = axes.ravel()
    for _ax in axes[len(strategies):]:
        _ax.set_visible(False)
    axes = axes[:len(strategies)]
    for ax, s in zip(axes, strategies):
        for i, (name, b) in enumerate(boards.items()):
            t = b[b.strategy == s].set_index("horizon").reindex(horizons)
            c = SERIES.get(name, INK2)
            if i == 0:
                ax.fill_between(x, t.ci_lo.to_numpy(float) * 100, t.ci_hi.to_numpy(float) * 100, color=c, alpha=0.10, linewidth=0)
            ax.plot(x, t["mean"].to_numpy(float) * 100, color=c, linewidth=2, marker="o", markersize=5,
                    markeredgecolor=SURFACE, markeredgewidth=1.2, linestyle="-" if i == 0 else "--",
                    label=f"{name} (n={int(t.n.max())})")
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_xticks(x, [str(h) if h != "exp" else "exp" for h in horizons])
        style(ax, STRATEGY_LABEL[s], "sessions after the filing session" if ax in axes[6:] else "",
              "mean P&L, % of spot" if ax in (axes[0], axes[3], axes[6]) else "")
    axes[0].legend(frameon=False, labelcolor=INK2, fontsize=8, loc="upper left")
    fig.suptitle(title, x=0.01, ha="left", color=INK, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


board = scoreboard(results)
print(f"In-sample · covered call · entry = post · n = {int(board.n.max())} events. * = 95% bootstrap CI excludes zero.")
display(fmt_board(board))
if "has_capped_call" in results.columns:
    uncapped = results[results["has_capped_call"] == False]
    capped = results[results["has_capped_call"] == True]
    if not uncapped.empty:
        print("\n--- Uncapped call events ---")
        display(fmt_board(scoreboard(uncapped)))
    if not capped.empty:
        print("\n--- Capped call events ---")
        display(fmt_board(scoreboard(capped)))

## 10 · Performance metrics — total P&L, Sharpe, max drawdown


In [ ]:
# Performance metrics: total P&L, Sharpe, max drawdown
def performance_metrics(res: pd.DataFrame, strategy: str = "covered_call") -> pd.DataFrame:
    """Compute total P&L, annualized Sharpe, and max drawdown by horizon."""
    post = res[res.entry == "post"].copy()
    if post.empty:
        return pd.DataFrame()
    
    # Dollar P&L per event
    post["dollar_pnl"] = post[strategy] * post["S_entry"]
    
    rows = []
    def _hkey(x):
        return (0, int(x)) if isinstance(x, (int, float)) else (1, str(x))
    for h in sorted(post["horizon"].dropna().unique(), key=_hkey):
        hdata = post[post["horizon"] == h].copy()
        if hdata.empty:
            continue
        
        # Total P&L
        total_pnl = hdata["dollar_pnl"].sum()
        total_return = hdata[strategy].sum()  # per $1 of spot
        
        # Sharpe ratio (annualized)
        mean_ret = hdata[strategy].mean()
        std_ret = hdata[strategy].std()
        avg_days = hdata["sessions_held"].mean() if "sessions_held" in hdata.columns else 10
        sharpe = (mean_ret / std_ret * np.sqrt(252 / avg_days)) if std_ret > 0 else np.nan
        
        # Max drawdown from cumulative returns ordered by exit_date
        hdata_sorted = hdata.sort_values("exit_date")
        cum_returns = (1 + hdata_sorted[strategy]).cumprod()
        peak = cum_returns.cummax()
        drawdown = (cum_returns - peak) / peak
        max_dd = drawdown.min() if not drawdown.empty else np.nan
        
        # Win rate
        win_rate = (hdata[strategy] > 0).mean()
        
        rows.append({
            "horizon": h,
            "n_events": len(hdata),
            "total_pnl_dollar": total_pnl,
            "total_return_per_unit": total_return,
            "mean_return": mean_ret,
            "std_return": std_ret,
            "sharpe_annualized": sharpe,
            "max_drawdown": max_dd,
            "win_rate": win_rate,
            "avg_days_held": avg_days,
        })
    
    return pd.DataFrame(rows)


metrics = performance_metrics(results, WINNER)
if not metrics.empty:
    print("Performance metrics by horizon:")
    display(metrics.round(4))
    
    # Plot cumulative returns
    post = results[results.entry == "post"].copy()
    post["dollar_pnl"] = post[WINNER] * post["S_entry"]
    post_sorted = post.sort_values("exit_date")
    cum_ret = (1 + post_sorted[WINNER]).cumprod()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cum_ret.values, color="#2a78d6", linewidth=2)
    ax.fill_between(range(len(cum_ret)), cum_ret.values, alpha=0.1, color="#2a78d6")
    ax.axhline(1, color="#898781", linewidth=0.8, linestyle="--")
    ax.set_title("Cumulative returns (covered call, post-entry)", color="#0b0b0b", fontsize=11, fontweight="bold", loc="left")
    ax.set_ylabel("Equity curve", color="#52514e", fontsize=9)
    ax.set_xlabel("Event # (ordered by exit date)", color="#52514e", fontsize=9)
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_facecolor("#fcfcfb")
    fig.patch.set_facecolor("#fcfcfb")
    plt.tight_layout()
    plt.show()
else:
    print("No post-entry events to compute metrics.")


## 9b · Stock return analysis


In [ ]:
# Stock return analysis around convertible offering
stock_returns = []
for pe in priced:
    S_entry = pe.stock(pe.t_0)
    for h in [1, 2, 3, 5, 10]:
        if pe.t_0 in CAL and CAL.get_loc(pe.t_0) + h < len(CAL):
            exit_day = CAL[CAL.get_loc(pe.t_0) + h]
            S_exit = pe.stock(exit_day)
            if not np.isnan(S_exit) and S_exit > 0:
                stock_returns.append({
                    "ticker": pe.ticker,
                    "event_date": pe.event_date,
                    "horizon": h,
                    "return": S_exit / S_entry - 1,
                })

if stock_returns:
    ret_df = pd.DataFrame(stock_returns)
    print("\nStock returns after convertible offering:")
    display(ret_df.groupby("horizon")["return"].agg(["mean", "median", "std", "count"]).mul(100).round(2))
else:
    print("No stock returns computed.")


In [ ]:
def run_study(tag, start, end, universe=None, buckets=None, max_events=None, label=None):
    """Pipeline: build events, filter, price, evaluate."""
    ev = build_convertible_events(start, end, UNIVERSE if universe is None else universe)
    ev = apply_filters(ev)  # screen moved post-pricing (attach_fulltext_flags below)
    if max_events:
        ev = ev.head(max_events).copy()
    pr, drops = price_events(ev, label=label or f"convertible {start}..{end}")
    pr = filter_liquid(pr)
    pr = attach_fulltext_flags(pr, ev) if pr else pr
    res = evaluate(pr)
    board = scoreboard(res) if not res.empty else pd.DataFrame(columns=["strategy", "horizon", "n", "mean", "ci_lo", "ci_hi", "median", "hit_rate"])
    return {"events": ev, "priced": pr, "dropped": drops, "results": res, "board": board}

def sample_placebo_events(ev: pd.DataFrame, n: int, start: str, end: str,
                            gap_days: int = PLACEBO_GAP_DAYS, seed: int = 7) -> pd.DataFrame:
    """Ordinary sessions for the same tickers (ticker-weighted), >= gap_days from any event."""
    rng = np.random.default_rng(seed)
    pool = CAL[(CAL >= pd.Timestamp(start)) & (CAL <= pd.Timestamp(end))]
    weights = ev.ticker.value_counts(normalize=True)
    rows = []
    for t in rng.choice(weights.index.to_numpy(), size=n, p=weights.to_numpy()):
        past = ev.loc[ev.ticker == t, "filing_date"]
        for _ in range(100):
            d = pd.Timestamp(rng.choice(pool))
            t_0 = session_on_or_after(d)
            if t_0 > pool[-1] or t_0 > LAST_SESSION - pd.Timedelta(days=5):
                continue
            if len(past) and bool((((past - t_0).abs().dt.days) < gap_days).any()):
                continue
            rows.append({"ticker": t, "filing_date": t_0, "t_pre": session_before(t_0),
                         "t_0": t_0, "has_capped_call": False, "supporting_text": "placebo"})
            break
    return pd.DataFrame(rows)


if RUN_PLACEBO and not events.empty:
    peb = sample_placebo_events(events, N_PLACEBO, STUDY_START, STUDY_END)
    ppr, _ = price_events(peb, label="placebo")
    pres = evaluate(ppr)
    pboard = scoreboard(pres)
    print(f"Placebo ({N_PLACEBO} ordinary sessions): n = {int(pboard.n.max())}")
    display(fmt_board(pboard))
    print("Event-minus-placebo edge (* = 95% CI on the difference excludes zero):")
    display(fmt_board(difference_board(results, pres), value="difference"))
elif events.empty:
    print("Placebo skipped: no in-sample events to mirror.")
else:
    print("Placebo disabled.")

def _split(flag: str, label: str):
    _sl = results[(results.entry == "post") & (results.horizon == 10)]
    print(f"{label} split at h=10, post entry (needs both groups present):")
    if not _sl.empty and _sl[flag].nunique() > 1:
        display(_sl.groupby(flag)[STRATEGIES].agg(["count", "mean", "median"]).round(4))
    else:
        print("only one group present -- no comparison possible yet.")


_split("has_capped_call", "Capped-call")
_split("has_repurchase", "Concurrent-repurchase")

print(f"Out-of-sample {OOS_START}..{OOS_END} (latest horizons may be unresolved and absent):")
oos = run_study(CONVERTIBLE_ITEM_CODES, OOS_START, OOS_END)
display(fmt_board(oos["board"]))